# GRU + BiLSTM — Fake News Detection

GRU feeds into a Bidirectional LSTM layer.

In [1]:
import os
import re
import json
import pickle

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import callbacks


In [2]:
try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))


## Hyperparameters
Edit these and re-run the notebook to try different settings.

In [3]:
PARAMS = dict(
    vocab_size=12000,
    max_len=40,
    embedding_dim=128,
    gru_units=96,
    lstm_units=64,
    dropout=0.3,
    learning_rate=1e-3,
    batch_size=64,
    epochs=12,
)


In [4]:
DATA_DIR = os.path.join(os.getcwd(), "data")
RESULTS_DIR = os.path.join(os.getcwd(), "results")
RANDOM_STATE = 42
os.makedirs(RESULTS_DIR, exist_ok=True)


## Load & preprocess data
Place `Fake.csv`/`True.csv` in `experiments/data/`, or this will auto-download via `kagglehub` (needs a Kaggle API token).

In [5]:
def get_dataset_paths():
    fake_path = os.path.join(DATA_DIR, "Fake.csv")
    true_path = os.path.join(DATA_DIR, "True.csv")
    if os.path.exists(fake_path) and os.path.exists(true_path):
        return fake_path, true_path
    # Falls back to auto-download from Kaggle if you have not placed the CSVs
    # in experiments/data/ yourself. Needs a Kaggle API token at
    # C:\Users\<you>\.kaggle\kaggle.json (or KAGGLE_USERNAME/KAGGLE_KEY env vars).
    import kagglehub
    kaggle_path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
    return os.path.join(kaggle_path, "Fake.csv"), os.path.join(kaggle_path, "True.csv")


def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", str(text).lower()).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)


fake_path, true_path = get_dataset_paths()
df_fake = pd.read_csv(fake_path)
df_true = pd.read_csv(true_path)
df_fake["label"] = 1  # fake
df_true["label"] = 0  # real
df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.dropna(subset=["title"]).reset_index(drop=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df["clean"] = df["title"].apply(clean_text)
df.head()


,title,text,subject,date,label,clean
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1,ben stein calls 9th circuit court committed co...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0,trump drops steve bannon national security cou...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0,puerto rico expects u lift jones act shipping ...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1,oops trump accidentally confirmed leaked israe...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0,donald trump heads scotland reopen golf resort


In [6]:
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df["clean"], df["label"], test_size=0.3,
    stratify=df["label"], random_state=RANDOM_STATE,
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.5,
    stratify=y_temp, random_state=RANDOM_STATE,
)

tokenizer = Tokenizer(num_words=PARAMS["vocab_size"], oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)


def to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=PARAMS["max_len"], padding="post", truncating="post")


X_train, y_train = to_padded(X_train_text), y_train.values
X_val, y_val = to_padded(X_val_text), y_val.values
X_test, y_test = to_padded(X_test_text), y_test.values

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)


train: (31428, 40)  val: (6735, 40)  test: (6735, 40)


## Build model

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Embedding(PARAMS["vocab_size"], PARAMS["embedding_dim"], input_length=PARAMS["max_len"]),
    GRU(PARAMS["gru_units"], dropout=PARAMS["dropout"], return_sequences=True),
    Bidirectional(LSTM(PARAMS["lstm_units"], dropout=PARAMS["dropout"])),
    Dropout(PARAMS["dropout"]),
    Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=Adam(learning_rate=PARAMS["learning_rate"]),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 40, 128)           1536000   
                                                                 
 gru (GRU)                   (None, 40, 96)            65088     
                                                                 
 bidirectional (Bidirection  (None, 128)               82432     
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1683649 (6.42 MB)
Trainable params: 1683649 (6.42 MB)
Non-trainable params: 0 (0.00 Byte)
_________________

## Train

In [8]:
model_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint(
        os.path.join(RESULTS_DIR, "gru_bilstm_best.h5"), monitor="val_loss", save_best_only=True,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=PARAMS["epochs"],
    batch_size=PARAMS["batch_size"],
    callbacks=model_callbacks,
    verbose=2,
)


Epoch 1/12


492/492 - 17s - loss: 0.1861 - accuracy: 0.9232 - val_loss: 0.1360 - val_accuracy: 0.9477 - 17s/epoch - 34ms/step
Epoch 2/12


c:\Users\ARKNOX\Documents\m.tech\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


492/492 - 14s - loss: 0.0789 - accuracy: 0.9715 - val_loss: 0.1280 - val_accuracy: 0.9520 - 14s/epoch - 28ms/step
Epoch 3/12
492/492 - 14s - loss: 0.0491 - accuracy: 0.9830 - val_loss: 0.1366 - val_accuracy: 0.9550 - 14s/epoch - 27ms/step
Epoch 4/12
492/492 - 14s - loss: 0.0319 - accuracy: 0.9893 - val_loss: 0.1730 - val_accuracy: 0.9488 - 14s/epoch - 28ms/step
Epoch 5/12
492/492 - 14s - loss: 0.0231 - accuracy: 0.9920 - val_loss: 0.1831 - val_accuracy: 0.9538 - 14s/epoch - 28ms/step


## Evaluate & save results

In [9]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "loss": float(loss),
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
}
print("gru_bilstm metrics:", metrics)

payload = {
    "model": "gru_bilstm",
    "params": PARAMS,
    "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
    "metrics": metrics,
}
with open(os.path.join(RESULTS_DIR, "gru_bilstm.json"), "w") as f:
    json.dump(payload, f, indent=2)

with open(os.path.join(RESULTS_DIR, "gru_bilstm_tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved gru_bilstm.json, gru_bilstm_best.h5, gru_bilstm_tokenizer.pkl in", RESULTS_DIR)


gru_bilstm metrics: {'loss': 0.12022873014211655, 'accuracy': 0.9545657015590201, 'precision': 0.96936095710534, 'recall': 0.9429463525404485, 'f1_score': 0.9559712230215828}
Saved gru_bilstm.json, gru_bilstm_best.h5, gru_bilstm_tokenizer.pkl in c:\Users\ARKNOX\Documents\m.tech\experiments\results
